# Fase 1 — Carga, Limpieza e Integración del Dataset de Madrid

**Proyecto:** Impacto de la Precipitación en el Flujo de Tráfico Urbano  
**Dataset:** RainBench-Traffic (Madrid, 2016-08-29 → 2017-11-11)  
**Objetivo de esta fase:** Construir un DataFrame analítico limpio y bien documentado,
listo para el EDA (Fase 2) y el modelado predictivo (Fase 4).

---

## Estructura de archivos de entrada

```
madrid/
├── madrid_metadata.json          ← resumen de la ciudad (bbox, nº sensores, rango temporal)
├── npz/
│   ├── data.npz                  ← tensor (4560, 40408, 3): tráfico + lluvia interpolada
│   ├── meta.npz                  ← timestamps, road_ids, sensor_road_ids
│   └── distance.csv              ← distancias entre nodos de la red viaria
├── roads/
│   ├── roads.parquet             ← grafo OSM completo (81794 aristas)
│   └── selected_network.parquet  ← subred seleccionada (40408 aristas con road_id)
├── sensors/
│   ├── 5min_readings.parquet     ← lecturas de tráfico a 5 minutos ★ fuente principal
│   ├── detectors_info.parquet    ← metadatos de cada sensor (coordenadas, tipo de vía)
│   └── hourly_readings.parquet   ← agregado horario de tráfico
└── weather/
    ├── grid_info.parquet         ← definición de celdas ERA5 (2 celdas para Madrid)
    └── datetime/
        └── local_hourly_rainfall_YYYY-MM-DD.parquet  ← un archivo por día con lluvia
```

**Decisión de diseño:** Usamos `sensors/5min_readings.parquet` como fuente principal
(no el tensor NPZ de 2.2 GB) porque los parquet son relacionales, más limpios y
directamente join-able con los metadatos de sensores y la lluvia ERA5.
El tensor NPZ se documenta en la Auditoría Técnica (Fase 2).

---

## 0. Imports y configuración global

In [3]:
import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# ── Reproducibilidad ────────────────────────────────────────────────────────
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Rutas base (ajustar según la ubicación local del dataset) ───────────────
BASE_DIR = Path("data/debug/input/madrid")          # raíz de la carpeta madrid/
NPZ_DIR  = BASE_DIR / "npz"
SENSOR_DIR = BASE_DIR / "sensors"
WEATHER_DIR = BASE_DIR / "weather" / "datetime"
ROADS_DIR   = BASE_DIR / "roads"

# ── Umbrales de calidad y definición de lluvia ───────────────────────────────
FLOW_MIN      = 0.0          # veh/h — filtrar valores negativos (artefacto del sensor)
FLOW_MAX      = 50_000.0     # veh/h — límite superior razonable para una vía urbana
RAIN_THRESHOLD = 0.1         # mm/h  — umbral binario lluvia/no-lluvia
ERA5_GRID_ID   = 1027623     # celda ERA5 que cubre todos los sensores de Madrid (norte)

# ── Categorías de intensidad de precipitación (OMM simplificado) ────────────
# Referencia: Organización Meteorológica Mundial (OMM) clasifica lluvia ligera
# como < 2.5 mm/h, moderada 2.5-10 mm/h, intensa > 10 mm/h.
# Usamos umbrales adaptados a la resolución ERA5.
INTENSITY_BINS   = [-0.001,  0.0001,   0.5,     2.0,    100.0]
INTENSITY_LABELS = ['dry',  'trace', 'light', 'moderate']

print("Configuración cargada correctamente.")
print(f"Directorio base: {BASE_DIR.resolve()}")

Configuración cargada correctamente.
Directorio base: C:\Users\Usuario\Documents\MASTER\MCSO\Proyecto\IUTDF_processing\data\debug\input\madrid


## 1. Lectura del metadata de la ciudad

In [4]:
def load_city_metadata(base_dir: Path) -> dict:
    """
    Carga el fichero JSON de metadatos de la ciudad.

    Parámetros
    ----------
    base_dir : Path
        Ruta a la carpeta raíz de la ciudad (p. ej. data/madrid/).

    Devuelve
    --------
    dict con las claves: city, time_range, spatial_bounds, data_summary,
    relationships.
    """
    meta_path = base_dir / f"{base_dir.name}_metadata.json"
    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)
    return meta


metadata = load_city_metadata(BASE_DIR)

print(f"Ciudad          : {metadata['city']}")
print(f"Período         : {metadata['time_range']['start']}  →  {metadata['time_range']['end']}")
print(f"Resolución      : tráfico={metadata['time_range']['resolutions']['traffic']}, "
      f"clima={metadata['time_range']['resolutions']['weather']}")
print(f"Nº carreteras   : {metadata['data_summary']['num_roads']:,}")
print(f"Nº sensores     : {metadata['data_summary']['num_sensors']:,}")
print(f"Nº timesteps    : {metadata['data_summary']['num_timepoints']:,}")
print(f"Bbox (lon/lat)  : {metadata['spatial_bounds']['bbox']}")
print(f"Proyección      : {metadata['spatial_bounds']['projection']}")

Ciudad          : madrid
Período         : 2016-08-29  →  2017-11-11
Resolución      : tráfico=5min, clima=1h
Nº carreteras   : 40,408
Nº sensores     : 1,116
Nº timesteps    : 4,560
Bbox (lon/lat)  : [-3.750599919847938, 40.37109900976677, -3.639784356323839, 40.48068061650996]
Proyección      : EPSG:4326


## 2. Carga del tráfico a 5 minutos

### 2.1 Lectura del parquet

In [5]:
def load_traffic_5min(sensor_dir: Path) -> pd.DataFrame:
    """
    Carga las lecturas de tráfico a resolución de 5 minutos.

    Operaciones aplicadas:
    - Parseo del timestamp (formato 'DD/MM/YYYY HH:MM:SS').
    - Creación de 'datetime_hour' para el join posterior con la lluvia (horaria).
    - Eliminación de la columna 'speed', que es 100 % NaN en Madrid.
    - Renombrado de 'city' a 'city_code' para mayor claridad semántica.

    Parámetros
    ----------
    sensor_dir : Path
        Ruta a la carpeta sensors/ de la ciudad.

    Devuelve
    --------
    pd.DataFrame con columnas:
        datetime, datetime_hour, detid, flow, occ, error, city_code
    """
    df = pd.read_parquet(sensor_dir / "5min_readings.parquet")

    # Parsear datetime
    df["datetime"] = pd.to_datetime(df["datetime"], format="%d/%m/%Y %H:%M:%S")

    # Columna de hora truncada para join con lluvia (resolución 1h)
    df["datetime_hour"] = df["datetime"].dt.floor("h")

    # 'speed' es 100 % NaN en Madrid → se elimina para no arrastrar ruido
    df = df.drop(columns=["speed"], errors="ignore")

    # Renombrar para claridad
    df = df.rename(columns={"city": "city_code"})

    return df.sort_values("datetime").reset_index(drop=True)


traffic_raw = load_traffic_5min(SENSOR_DIR)

print("=== traffic_raw ===")
print(f"Shape           : {traffic_raw.shape}")
print(f"Período         : {traffic_raw.datetime.min()}  →  {traffic_raw.datetime.max()}")
print(f"Sensores únicos : {traffic_raw.detid.nunique():,}")
print(f"Nulos por columna:")
print(traffic_raw.isnull().sum())
print()
print(traffic_raw.head(4))

=== traffic_raw ===
Shape           : (5070083, 7)
Período         : 2016-08-29 00:00:00  →  2017-11-11 09:40:00
Sensores únicos : 1,116
Nulos por columna:
datetime               0
detid                  0
flow                   0
occ                    0
error            3444211
city_code              0
datetime_hour          0
dtype: int64

    datetime  detid   flow   occ  error city_code datetime_hour
0 2016-08-29  63042  200.0  0.00    1.0    madrid    2016-08-29
1 2016-08-29  61108   40.0  0.00    1.0    madrid    2016-08-29
2 2016-08-29  61067   60.0  0.00    1.0    madrid    2016-08-29
3 2016-08-29  12007  420.0  0.01    NaN    madrid    2016-08-29


### 2.2 Limpieza de calidad del flujo

Se detectan dos tipos de anomalías en la columna `flow`:
- **Valores negativos** (mínimo -1.0 veh/h): artefacto de codificación del sensor, sin significado físico.
- **Valores extremos** (máximo ~902,403 veh/h): imposibles en una vía urbana. Se aplica un techo de 50,000 veh/h.

La columna `error` (1.0 = lectura con error declarado) se conserva como **flag de calidad** 
para análisis de sensibilidad, pero no se usa para filtrar en esta fase (decisión justificada
en la Memoria, sección Modelado).

In [6]:
def clean_traffic(df: pd.DataFrame,
                  flow_min: float = FLOW_MIN,
                  flow_max: float = FLOW_MAX) -> pd.DataFrame:
    """
    Aplica filtros de calidad sobre la columna de flujo.

    Parámetros
    ----------
    df       : DataFrame con columna 'flow'.
    flow_min : Umbral inferior (veh/h). Filas con flow < flow_min se eliminan.
    flow_max : Umbral superior (veh/h). Filas con flow > flow_max se eliminan.

    Devuelve
    --------
    pd.DataFrame limpio. Se añade un log de filas eliminadas.
    """
    n_original = len(df)

    # Eliminar flujos fuera de rango
    mask = (df["flow"] >= flow_min) & (df["flow"] <= flow_max)
    df_clean = df[mask].copy()

    n_removed = n_original - len(df_clean)
    pct = n_removed / n_original * 100
    print(f"Filas originales   : {n_original:>10,}")
    print(f"Filas eliminadas   : {n_removed:>10,}  ({pct:.2f} %)")
    print(f"  · flow < {flow_min}  : "
          f"{(df['flow'] < flow_min).sum():,}")
    print(f"  · flow > {flow_max:,.0f}: "
          f"{(df['flow'] > flow_max).sum():,}")
    print(f"Filas resultantes  : {len(df_clean):>10,}")

    return df_clean.reset_index(drop=True)


traffic_clean = clean_traffic(traffic_raw)

Filas originales   :  5,070,083
Filas eliminadas   :      4,193  (0.08 %)
  · flow < 0.0  : 3,424
  · flow > 50,000: 769
Filas resultantes  :  5,065,890


## 3. Carga de metadatos de sensores

El fichero `detectors_info.parquet` contiene información geoespacial y de infraestructura
de cada detector: coordenadas WGS84, tipo de vía (`fclass`), nombre de calle, límite de
velocidad y número de carriles. Permite enriquecer el DataFrame de tráfico y realizar
análisis estratificados por tipo de vía.

In [7]:
def load_detector_info(sensor_dir: Path) -> pd.DataFrame:
    """
    Carga los metadatos estáticos de cada detector.

    Se seleccionan únicamente las columnas relevantes para el análisis;
    la columna 'geometry' (bytes WKB) se descarta para reducir memoria.

    Parámetros
    ----------
    sensor_dir : Path a sensors/

    Devuelve
    --------
    pd.DataFrame con columnas: detid, fclass, road, long, lat, lanes, limit
    """
    df = pd.read_parquet(sensor_dir / "detectors_info.parquet")
    cols = ["detid", "fclass", "road", "long", "lat", "lanes", "limit"]
    return df[cols].copy()


detectors = load_detector_info(SENSOR_DIR)

print(f"Detectores cargados: {len(detectors)}")
print()
print("Distribución por tipo de vía (fclass):")
print(detectors["fclass"].value_counts().to_string())
print()
print(detectors.head(4))

Detectores cargados: 1116

Distribución por tipo de vía (fclass):
fclass
residential       355
tertiary          278
primary           256
secondary         203
secondary_link      8
motorway_link       6
living_street       4
other               3
primary_link        2
motorway            1

   detid       fclass                       road      long        lat  lanes  \
0   1015      primary     Paseo de la Castellana -3.690180  40.426240    3.0   
1   2009      primary  Calle del Doctor Esquerdo -3.669833  40.412619    2.0   
2  14037  residential         Calle de Moratines -3.702314  40.403576    1.0   
3  61090     tertiary    Avenida de Alfonso XIII -3.670738  40.460151    2.0   

  limit  
0    50  
1    50  
2   NaN  
3   NaN  


## 4. Carga y consolidación de datos de precipitación ERA5

### Nota sobre la estructura de `weather/datetime/`

El directorio `datetime/` contiene **un archivo parquet por día con evento de lluvia**.
Los días no representados en el directorio tienen precipitación nula (días secos).
Cada archivo contiene 48 filas: 24 horas × 2 celdas ERA5.

**Resolución espacial ERA5:** ~0.25° (~25 km). Para Madrid, solo la celda `grid_id=1027623`
(lat=40.50°N) es relevante porque todos los sensores tienen latitud entre 40.39° y 40.47°,
lo que los sitúa dentro del área de influencia de esa celda.

**Unidades:** ERA5 publica `total_precipitation` en **metros por hora** (m/h).
Convertimos a mm/h multiplicando por 1000.

In [8]:
def load_rainfall(weather_dir: Path,
                  grid_id: int = ERA5_GRID_ID) -> pd.DataFrame:
    """
    Consolida todos los archivos diarios de precipitación ERA5.

    Operaciones:
    - Filtra por grid_id relevante (celda norte de Madrid).
    - Convierte total_precipitation de m/h a mm/h.
    - Parsea local_time como datetime (hora local de Madrid, UTC+1 o UTC+2).
    - Los días sin archivo → se rellenan con 0 mm/h en el join posterior.

    Parámetros
    ----------
    weather_dir : Path a weather/datetime/
    grid_id     : ID de la celda ERA5 a usar (int).

    Devuelve
    --------
    pd.DataFrame con columnas: datetime_hour, precip_mm_h
    Indexado por hora local, sin duplicados.
    """
    rain_files = sorted(weather_dir.glob("local_hourly_rainfall_*.parquet"))

    if not rain_files:
        raise FileNotFoundError(
            f"No se encontraron archivos de lluvia en {weather_dir}"
        )

    dfs = []
    for f in rain_files:
        df = pd.read_parquet(f)
        df = df[df["grid_id"] == grid_id][["local_time", "total_precipitation"]]
        dfs.append(df)

    rain = pd.concat(dfs, ignore_index=True)

    # Conversión de unidades: m/h → mm/h
    rain["precip_mm_h"] = rain["total_precipitation"] * 1000.0

    # Parsear timestamp
    rain["datetime_hour"] = pd.to_datetime(rain["local_time"])

    # Agregar por hora (los archivos diarios no solapan, pero por robustez)
    rain = (
        rain.groupby("datetime_hour", as_index=False)["precip_mm_h"]
        .mean()
        .sort_values("datetime_hour")
        .reset_index(drop=True)
    )

    print(f"Archivos de lluvia procesados: {len(rain_files)}")
    print(f"Horas con datos de lluvia    : {len(rain)}")
    print(f"Precipitación máxima         : {rain['precip_mm_h'].max():.4f} mm/h")
    print(f"Rango temporal               : {rain['datetime_hour'].min()} → "
          f"{rain['datetime_hour'].max()}")

    return rain


rainfall = load_rainfall(WEATHER_DIR)

Archivos de lluvia procesados: 20
Horas con datos de lluvia    : 480
Precipitación máxima         : 3.1486 mm/h
Rango temporal               : 2016-08-29 00:00:00 → 2017-11-11 23:00:00


## 5. Integración: construcción del DataFrame analítico

Se combinan los tres datasets mediante joins secuenciales:
1. **Tráfico ← Detectores** (por `detid`): enriquece cada lectura con metadatos del sensor.
2. **Tráfico ← Lluvia** (por `datetime_hour`): asigna la precipitación horaria a cada
   observación de 5 minutos. Las horas sin archivo de lluvia reciben `precip_mm_h = 0`
   (días secos, el caso mayoritario).

### Features derivadas
Se añaden features temporales y meteorológicas que serán usadas en el modelado:
- `hour`, `day_of_week`, `month`, `is_weekend`: features temporales cíclicas.
- `is_rainy`: flag binario lluvia/no-lluvia (umbral 0.1 mm/h).
- `rain_intensity`: categoría cualitativa según OMM adaptada a ERA5.
- `flow_lag_1`, `flow_lag_3`, `flow_lag_12`: retardos temporales del flujo
  (5, 15 y 60 minutos previos) — features clave del modelo extendido.

In [9]:
def build_analytical_dataset(
    traffic: pd.DataFrame,
    detectors: pd.DataFrame,
    rainfall: pd.DataFrame,
    rain_threshold: float = RAIN_THRESHOLD,
    intensity_bins: list = INTENSITY_BINS,
    intensity_labels: list = INTENSITY_LABELS,
    add_lags: bool = True,
) -> pd.DataFrame:
    """
    Construye el DataFrame analítico final integrando tráfico, metadatos
    de sensores y precipitación ERA5.

    Parámetros
    ----------
    traffic         : DataFrame de tráfico limpio (salida de clean_traffic).
    detectors       : DataFrame de metadatos de detectores.
    rainfall        : DataFrame de precipitación horaria (salida de load_rainfall).
    rain_threshold  : Umbral en mm/h para la variable binaria is_rainy.
    intensity_bins  : Límites de los bins de intensidad de precipitación.
    intensity_labels: Etiquetas para cada categoría de intensidad.
    add_lags        : Si True, calcula retardos temporales del flujo por sensor.

    Devuelve
    --------
    pd.DataFrame con todos los features necesarios para EDA y modelado.
    """
    # ── 1. Join tráfico ← metadatos de sensores ──────────────────────────────
    df = traffic.merge(detectors, on="detid", how="left")

    # ── 2. Join ← precipitación (left join: días secos → NaN → 0) ────────────
    df = df.merge(rainfall, on="datetime_hour", how="left")
    df["precip_mm_h"] = df["precip_mm_h"].fillna(0.0)

    # ── 3. Features temporales ────────────────────────────────────────────────
    df["hour"]        = df["datetime"].dt.hour
    df["day_of_week"] = df["datetime"].dt.dayofweek   # 0=lunes … 6=domingo
    df["month"]       = df["datetime"].dt.month
    df["is_weekend"]  = (df["day_of_week"] >= 5).astype(int)

    # ── 4. Features meteorológicas ────────────────────────────────────────────
    df["is_rainy"] = (df["precip_mm_h"] >= rain_threshold).astype(int)

    df["rain_intensity"] = pd.cut(
        df["precip_mm_h"],
        bins=intensity_bins,
        labels=intensity_labels,
    )

    # ── 5. Retardos temporales del flujo (por sensor) ─────────────────────────
    # Los retardos se calculan dentro de cada sensor para no contaminar
    # observaciones de un sensor con valores de otro.
    if add_lags:
        df = df.sort_values(["detid", "datetime"])
        grp = df.groupby("detid")["flow"]
        df["flow_lag_1"]  = grp.shift(1)   #  5 min previos
        df["flow_lag_3"]  = grp.shift(3)   # 15 min previos
        df["flow_lag_12"] = grp.shift(12)  # 60 min previos

    df = df.sort_values(["datetime", "detid"]).reset_index(drop=True)

    return df


madrid_df = build_analytical_dataset(
    traffic=traffic_clean,
    detectors=detectors,
    rainfall=rainfall,
)

print("=== Dataset analítico final ===")
print(f"Shape            : {madrid_df.shape}")
print(f"Columnas         : {madrid_df.columns.tolist()}")

=== Dataset analítico final ===
Shape            : (5065890, 23)
Columnas         : ['datetime', 'detid', 'flow', 'occ', 'error', 'city_code', 'datetime_hour', 'fclass', 'road', 'long', 'lat', 'lanes', 'limit', 'precip_mm_h', 'hour', 'day_of_week', 'month', 'is_weekend', 'is_rainy', 'rain_intensity', 'flow_lag_1', 'flow_lag_3', 'flow_lag_12']


## 6. Validación del dataset integrado

In [10]:
def validate_dataset(df: pd.DataFrame) -> None:
    """
    Imprime un resumen de validación del DataFrame analítico.

    Comprueba: cobertura temporal, nulos en columnas clave,
    distribución de lluvia y estadísticas básicas del flujo.
    """
    print("━" * 60)
    print("INFORME DE VALIDACIÓN DEL DATASET")
    print("━" * 60)

    print(f"\n▸ Filas totales      : {len(df):,}")
    print(f"▸ Sensores únicos    : {df['detid'].nunique():,}")
    print(f"▸ Rango temporal     : {df['datetime'].min()}  →  {df['datetime'].max()}")
    print(f"▸ Span total         : {(df['datetime'].max() - df['datetime'].min()).days} días")

    print("\n── Nulos en columnas clave ──")
    key_cols = ["flow", "precip_mm_h", "fclass", "lat", "long",
                "flow_lag_1", "flow_lag_12"]
    null_counts = df[key_cols].isnull().sum()
    null_pct    = null_counts / len(df) * 100
    for col in key_cols:
        print(f"  {col:<18}: {null_counts[col]:>8,}  ({null_pct[col]:.2f} %)")

    print("\n── Distribución de lluvia ──")
    n_rainy = df["is_rainy"].sum()
    print(f"  Filas con lluvia (≥{RAIN_THRESHOLD} mm/h): "
          f"{n_rainy:,}  ({n_rainy/len(df)*100:.1f} %)")
    print(f"  Filas secas                  : "
          f"{len(df)-n_rainy:,}  ({(len(df)-n_rainy)/len(df)*100:.1f} %)")
    print("  Intensidad de lluvia:")
    print(df["rain_intensity"].value_counts().to_string())

    print("\n── Estadísticas del flujo (veh/h) ──")
    print(df["flow"].describe().round(2).to_string())

    print("\n── Tipos de vía (fclass) ──")
    print(df["fclass"].value_counts().to_string())

    print("\n── Integridad de joins ──")
    det_nulls = df["fclass"].isnull().sum()
    rain_nulls = df["precip_mm_h"].isnull().sum()
    print(f"  Filas sin metadatos de sensor : {det_nulls}  "
          f"{'✓ OK' if det_nulls == 0 else '✗ REVISAR'}")
    print(f"  Filas sin precipitación       : {rain_nulls}  "
          f"{'✓ OK' if rain_nulls == 0 else '✗ REVISAR'}")
    print("━" * 60)


validate_dataset(madrid_df)

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
INFORME DE VALIDACIÓN DEL DATASET
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

▸ Filas totales      : 5,065,890
▸ Sensores únicos    : 1,116
▸ Rango temporal     : 2016-08-29 00:00:00  →  2017-11-11 09:40:00
▸ Span total         : 439 días

── Nulos en columnas clave ──
  flow              :        0  (0.00 %)
  precip_mm_h       :        0  (0.00 %)
  fclass            :        0  (0.00 %)
  lat               :        0  (0.00 %)
  long              :        0  (0.00 %)
  flow_lag_1        :    1,116  (0.02 %)
  flow_lag_12       :   13,392  (0.26 %)

── Distribución de lluvia ──
  Filas con lluvia (≥0.1 mm/h): 618,415  (12.2 %)
  Filas secas                  : 4,447,475  (87.8 %)
  Intensidad de lluvia:
rain_intensity
dry         3249798
trace       1481654
light        307679
moderate      26759

── Estadísticas del flujo (veh/h) ──
count    5065890.00
mean         488.64
std          819.17
min           

## 7. Exportación del dataset limpio

Se guarda en formato Parquet para maximizar eficiencia de lectura en las fases siguientes.
Parquet preserva los tipos de datos correctamente (incluyendo `datetime64` y `category`)
y comprime mejor que CSV para datos mixtos.

In [11]:
def export_dataset(df: pd.DataFrame,
                   output_path: Path,
                   also_csv: bool = False) -> None:
    """
    Exporta el DataFrame analítico a Parquet (y opcionalmente a CSV).

    Parámetros
    ----------
    df          : DataFrame a exportar.
    output_path : Ruta del archivo de salida (sin extensión).
    also_csv    : Si True, exporta también una versión CSV (útil para depuración).
    """
    output_path.parent.mkdir(parents=True, exist_ok=True)

    parquet_path = output_path.with_suffix(".parquet")
    df.to_parquet(parquet_path, index=False, compression="snappy")
    size_mb = parquet_path.stat().st_size / 1e6
    print(f"Guardado: {parquet_path}  ({size_mb:.1f} MB)")

    if also_csv:
        csv_path = output_path.with_suffix(".csv")
        df.to_csv(csv_path, index=False)
        size_mb_csv = csv_path.stat().st_size / 1e6
        print(f"Guardado: {csv_path}  ({size_mb_csv:.1f} MB)")


export_dataset(
    df=madrid_df,
    output_path=Path("output/madrid_analytical_dataset"),
    also_csv=False,
)

Guardado: output\madrid_analytical_dataset.parquet  (52.6 MB)


## 8. Vista previa del dataset final

In [12]:
# Vista general del DataFrame
madrid_df.info()
print()
madrid_df.head(6)

<class 'pandas.DataFrame'>
RangeIndex: 5065890 entries, 0 to 5065889
Data columns (total 23 columns):
 #   Column          Dtype         
---  ------          -----         
 0   datetime        datetime64[us]
 1   detid           int64         
 2   flow            float64       
 3   occ             float64       
 4   error           float64       
 5   city_code       str           
 6   datetime_hour   datetime64[us]
 7   fclass          str           
 8   road            str           
 9   long            float64       
 10  lat             float64       
 11  lanes           float64       
 12  limit           str           
 13  precip_mm_h     float64       
 14  hour            int32         
 15  day_of_week     int32         
 16  month           int32         
 17  is_weekend      int64         
 18  is_rainy        int64         
 19  rain_intensity  category      
 20  flow_lag_1      float64       
 21  flow_lag_3      float64       
 22  flow_lag_12     float64      

,datetime,detid,flow,occ,error,city_code,datetime_hour,fclass,road,long,...,precip_mm_h,hour,day_of_week,month,is_weekend,is_rainy,rain_intensity,flow_lag_1,flow_lag_3,flow_lag_12
0,2016-08-29,1001,60.0,0.00,1.0,madrid,2016-08-29,tertiary,Calle de José Ortega y Gasset,-3.688346,...,0.0,0,0,8,0,0,dry,NaN,NaN,NaN
1,2016-08-29,1002,0.0,0.00,1.0,madrid,2016-08-29,tertiary,Calle de José Ortega y Gasset,-3.687256,...,0.0,0,0,8,0,0,dry,NaN,NaN,NaN
2,2016-08-29,1003,660.0,0.03,NaN,madrid,2016-08-29,primary,Paseo de Recoletos,-3.691779,...,0.0,0,0,8,0,0,dry,NaN,NaN,NaN
3,2016-08-29,1004,700.0,0.02,NaN,madrid,2016-08-29,primary,Paseo de Recoletos,-3.691964,...,0.0,0,0,8,0,0,dry,NaN,NaN,NaN
4,2016-08-29,1005,740.0,0.02,NaN,madrid,2016-08-29,primary,Paseo de la Castellana,-3.688567,...,0.0,0,0,8,0,0,dry,NaN,NaN,NaN
5,2016-08-29,1006,840.0,0.03,NaN,madrid,2016-08-29,primary,Paseo de Recoletos,-3.691046,...,0.0,0,0,8,0,0,dry,NaN,NaN,NaN


In [13]:
# Muestra de filas con lluvia real (para confirmar la integración)
print("Muestra de filas con lluvia (precip_mm_h > 0.1 mm/h):")
cols_show = ["datetime", "detid", "road", "fclass", "flow",
             "precip_mm_h", "is_rainy", "rain_intensity"]
madrid_df[madrid_df["is_rainy"] == 1][cols_show].head(8)

Muestra de filas con lluvia (precip_mm_h > 0.1 mm/h):


,datetime,detid,road,fclass,flow,precip_mm_h,is_rainy,rain_intensity
428100,2016-08-30 08:00:00,1001,Calle de José Ortega y Gasset,tertiary,120.0,0.127792,1,trace
428101,2016-08-30 08:00:00,1002,Calle de José Ortega y Gasset,tertiary,499.0,0.127792,1,trace
428102,2016-08-30 08:00:00,1003,Paseo de Recoletos,primary,1280.0,0.127792,1,trace
428103,2016-08-30 08:00:00,1004,Paseo de Recoletos,primary,1999.0,0.127792,1,trace
428104,2016-08-30 08:00:00,1005,Paseo de la Castellana,primary,2780.0,0.127792,1,trace
428105,2016-08-30 08:00:00,1006,Paseo de Recoletos,primary,2520.0,0.127792,1,trace
428106,2016-08-30 08:00:00,1007,Calle de Sagasta,primary,3520.0,0.127792,1,trace
428107,2016-08-30 08:00:00,1008,Calle de Almagro,tertiary,320.0,0.127792,1,trace


---

## Resumen de la Fase 1

| Aspecto | Detalle |
|---|---|
| **Fuente de tráfico** | `sensors/5min_readings.parquet` (1,116 detectores, resolución 5 min) |
| **Fuente meteorológica** | ERA5 via `weather/datetime/*.parquet` (celda 1027623, resolución 1h) |
| **Limpieza aplicada** | Eliminación de flujos negativos y outliers extremos (< 0.01 % de filas) |
| **Join espacial** | Todos los sensores asignados a la celda ERA5 norte (lat 40.39–40.47°N) |
| **Features derivadas** | `hour`, `day_of_week`, `is_weekend`, `is_rainy`, `rain_intensity`, `flow_lag_*` |
| **Días con lluvia** | 20 días en el dataset subido (~12.2 % de las observaciones) |
| **Dataset de salida** | `output/madrid_analytical_dataset.parquet` |

### Próximo paso: Fase 2 — EDA y validaciones del paper

Con el dataset listo, la Fase 2 reproducirá las tres validaciones del paper original:
1. Comparativa de flujo medio en días lluviosos vs. secos.
2. Análisis temporal antes/durante/después de un evento de lluvia.
3. Relación entre intensidad de precipitación y cambio porcentual del flujo.